# Track 1 — Can It Play DOOM? (Colab A100 Hill-Climbing Pipeline)

End-to-end Google Colab A100 notebook for `hill-climb/ego-memory-v2`:
- Egocentric 8-channel odometric spatial memory (`submission/ego_map.py` + `SpatialMapEncoder`)
- Training-only geodesic PBRS (`training/pbrs.py`), rebalanced combat/progression rewards, and 5-family procedural map variations
- Adaptive per-map curriculum + clipped Huber value loss + L/R mirror augmentation + smooth entropy/LR annealing
- Iterative Round 1 -> Round 2 -> Round 3 training and held-out Dev + Stress evaluation (`training/eval.py`)

In [ ]:
!nvidia-smi
!nproc

# Mount Google Drive for persistent checkpoints and eval JSONs across Colab sessions
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/RohitBharadwaj-rvu/track-1.git"
BRANCH = "hill-climb/ego-memory-v2"
REPO_DIR = Path("/content/track1")

if not REPO_DIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git fetch origin && git checkout {BRANCH} && git pull --ff-only

!bash training/colab_setup.sh

In [ ]:
%cd /content/track1
!.venv/bin/pytest -q

# Sweep worker counts on this A100 runtime to confirm optimal --num-envs
!.venv/bin/python -m training.train --bench-workers 4,8,12 --rollout-steps 256 --use-ego-map --device cuda

In [ ]:
%cd /content/track1
import os
from pathlib import Path

RUN_ROOT = Path("/content/drive/MyDrive/track1_runs/ego_v2") if Path("/content/drive/MyDrive").exists() else Path("/content/track1/training/runs/ego_v2")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
NUM_ENVS = 12  # adjust if the worker sweep above shows 8 is faster

# Round 1: Foundation + Progression (1.0M decisions on Train + 40 procedural maps)
!.venv/bin/python -m training.train \
    --arch telemetry_gru_v1 --action-head factorized --use-ego-map \
    --split train --procedural-per-family 8 --procedural-dir {RUN_ROOT}/proc_r1 \
    --num-envs {NUM_ENVS} --vec-mode subproc --rollout-steps 256 --chunk-len 64 \
    --num-minibatches 4 --update-epochs 4 --total-steps 1000000 --seed 0 --device cuda \
    --lr 3e-4 --min-lr-frac 0.30 --ent-coef 0.015 --ent-coef-end 0.006 --gamma 0.995 \
    --pbrs-scale 3.0 --curiosity-coef 0.0 --clip-vloss --huber-vloss --huber-delta 5.0 \
    --mirror-prob 0.5 --adaptive-weights \
    --checkpoint-dir {RUN_ROOT} --export-path {RUN_ROOT}/policy_r1.pt --save-interval 25

# Evaluate Dev (stochastic vs deterministic) + Stress navigation suite
!DOOM_AGENT_WEIGHTS={RUN_ROOT}/policy_r1.pt .venv/bin/python -m training.eval \
    --split dev --seeds 0 1 --device cuda --compare-determinism
!DOOM_AGENT_WEIGHTS={RUN_ROOT}/policy_r1.pt .venv/bin/python -m training.eval \
    --split dev --seeds 0 1 --device cuda --eval-mode deterministic --out {RUN_ROOT}/eval_dev_r1.json
!DOOM_AGENT_WEIGHTS={RUN_ROOT}/policy_r1.pt .venv/bin/python -m training.eval \
    --suite stress --seeds 0 1 2 --device cuda --eval-mode deterministic --out {RUN_ROOT}/eval_stress_r1.json

In [ ]:
%cd /content/track1
from pathlib import Path

RUN_ROOT = Path("/content/drive/MyDrive/track1_runs/ego_v2") if Path("/content/drive/MyDrive").exists() else Path("/content/track1/training/runs/ego_v2")
NUM_ENVS = 12

# Round 2: Scale generalization + Tier 3-4 mastery (+1.5M decisions via --resume, lower PBRS)
!.venv/bin/python -m training.train \
    --resume {RUN_ROOT}/latest_train_state.pt \
    --split train --procedural-per-family 12 --procedural-dir {RUN_ROOT}/proc_r2 \
    --num-envs {NUM_ENVS} --vec-mode subproc --rollout-steps 256 --chunk-len 64 \
    --num-minibatches 4 --update-epochs 4 --total-steps 1500000 --seed 100 --device cuda \
    --lr 2e-4 --min-lr-frac 0.25 --ent-coef 0.008 --ent-coef-end 0.004 --gamma 0.995 \
    --pbrs-scale 1.5 --curiosity-coef 0.0 --clip-vloss --huber-vloss --huber-delta 5.0 \
    --mirror-prob 0.5 --adaptive-weights \
    --checkpoint-dir {RUN_ROOT} --export-path {RUN_ROOT}/policy_r2.pt --save-interval 25

!DOOM_AGENT_WEIGHTS={RUN_ROOT}/policy_r2.pt .venv/bin/python -m training.eval \
    --split dev --seeds 0 1 --device cuda --compare-determinism
!DOOM_AGENT_WEIGHTS={RUN_ROOT}/policy_r2.pt .venv/bin/python -m training.eval \
    --split dev --seeds 0 1 --device cuda --eval-mode deterministic --out {RUN_ROOT}/eval_dev_r2.json
!DOOM_AGENT_WEIGHTS={RUN_ROOT}/policy_r2.pt .venv/bin/python -m training.eval \
    --suite stress --seeds 0 1 2 --device cuda --eval-mode deterministic --out {RUN_ROOT}/eval_stress_r2.json

In [ ]:
%cd /content/track1
from pathlib import Path

RUN_ROOT = Path("/content/drive/MyDrive/track1_runs/ego_v2") if Path("/content/drive/MyDrive").exists() else Path("/content/track1/training/runs/ego_v2")
NUM_ENVS = 12

# Round 3: Fine-tuning with low PBRS + low entropy (+1.5M decisions)
!.venv/bin/python -m training.train \
    --resume {RUN_ROOT}/latest_train_state.pt \
    --split train --procedural-per-family 12 --procedural-dir {RUN_ROOT}/proc_r3 \
    --num-envs {NUM_ENVS} --vec-mode subproc --rollout-steps 256 --chunk-len 64 \
    --num-minibatches 4 --update-epochs 4 --total-steps 1500000 --seed 200 --device cuda \
    --lr 1.5e-4 --min-lr-frac 0.15 --ent-coef 0.005 --ent-coef-end 0.002 --gamma 0.995 \
    --pbrs-scale 0.5 --curiosity-coef 0.0 --clip-vloss --huber-vloss --huber-delta 5.0 \
    --mirror-prob 0.5 --adaptive-weights \
    --checkpoint-dir {RUN_ROOT} --export-path {RUN_ROOT}/policy_r3.pt --save-interval 25

!DOOM_AGENT_WEIGHTS={RUN_ROOT}/policy_r3.pt .venv/bin/python -m training.eval \
    --split dev --seeds 0 1 --device cuda --compare-determinism
!DOOM_AGENT_WEIGHTS={RUN_ROOT}/policy_r3.pt .venv/bin/python -m training.eval \
    --suite stress --seeds 0 1 2 --device cuda --eval-mode deterministic --out {RUN_ROOT}/eval_stress_r3.json

# Set USE_DETERMINISTIC based on --compare-determinism winner above, then promote & validate:
USE_DETERMINISTIC = True
!.venv/bin/python -c "import dataclasses; from submission.policy import load_checkpoint, save_checkpoint; model, cfg, meta = load_checkpoint('{RUN_ROOT}/policy_r3.pt', device='cpu'); cfg = dataclasses.replace(cfg, eval_deterministic={USE_DETERMINISTIC}); save_checkpoint('submission/weights/policy.pt', model, cfg, meta=meta); print('Saved promoted submission/weights/policy.pt with eval_deterministic =', cfg.eval_deterministic)"

!.venv/bin/python -m training.eval --split dev --seeds 0 1 --device cuda --out training/results/eval_dev_final.json
!.venv/bin/python -m training.eval --split dev --seeds 0 1 2 --device cuda
!.venv/bin/python -m training.eval --split final --seeds 0 1 --device cuda
!.venv/bin/python validate.py --device cuda --map maps/two_rooms.wad --map maps/two_rooms.wad